# Google Pay Expense Sharing

This project shows how shared expenses can be divided between friends.

Here we use three friends: **Alice, Bob and Carol**.
The data is kept simple so that the calculation is easy to understand.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Read the expense data

In [ ]:
df = pd.read_csv('easy_expenses.csv')
df

### What the columns mean

- **payer** → the person who paid
- **amount** → the bill amount
- **participants** → people sharing the bill
- **split_weights** → how the bill is divided
- **status** → paid, refund or unpaid

## 3. Check the data

In [ ]:
print('Number of expenses:', len(df))
print('Number of columns:', len(df.columns))

print('\nMissing values:')
print(df.isnull().sum())

## 4. Prepare the data

In [ ]:
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['status'] = df['status'].str.lower()

df[['id', 'description', 'payer', 'amount', 'status']]

## 5. Find the people

In [ ]:
people = set()

for value in df['participants'].dropna():
    people.update(value.split('|'))

people = sorted(people)
print('People:', people)

## 6. Calculate each person's share

For equal sharing, each person gets the same part of the bill.

For example, a ₹900 lunch shared by 3 people means:

**₹900 ÷ 3 = ₹300 each**.

In [ ]:
balance = {person: 0.0 for person in people}
paid = {person: 0.0 for person in people}
share = {person: 0.0 for person in people}
pending = []

for _, row in df.iterrows():
    amount = float(row['amount'])
    people_in_bill = row['participants'].split('|')
    weights = [float(x) for x in row['split_weights'].split('|')]
    total_weight = sum(weights)

    # Unpaid bills are kept pending.
    if row['status'] == 'unpaid':
        pending.append(row['id'])
        continue

    payer = row['payer']

    if row['status'] == 'paid':
        balance[payer] += amount
        paid[payer] += amount

        for person, weight in zip(people_in_bill, weights):
            each_share = amount * weight / total_weight
            balance[person] -= each_share
            share[person] += each_share

    # A refund gives money back to the original payer.
    elif row['status'] == 'refund':
        balance[payer] += amount

        for person, weight in zip(people_in_bill, weights):
            each_share = amount * weight / total_weight
            balance[person] += each_share
            share[person] -= each_share

print('Final balance:')
for person in people:
    print(person, 'Rs.', round(balance[person], 2))

## 7. Who should pay and who should receive?

In [ ]:
for person in people:
    amount = round(balance[person], 2)

    if amount > 0:
        print(person, 'should receive Rs.', amount)
    elif amount < 0:
        print(person, 'should pay Rs.', abs(amount))
    else:
        print(person, 'is settled')

## 8. Settlement plan

Now we match the people who have to pay with the people who should receive money.

In [ ]:
payers = [[p, -amount] for p, amount in balance.items() if amount < -0.01]
receivers = [[p, amount] for p, amount in balance.items() if amount > 0.01]

i = 0
j = 0

while i < len(payers) and j < len(receivers):
    payer = payers[i][0]
    receiver = receivers[j][0]
    amount = round(min(payers[i][1], receivers[j][1]), 2)

    print(payer, 'pays', receiver, 'Rs.', amount)

    payers[i][1] -= amount
    receivers[j][1] -= amount

    if payers[i][1] < 0.01:
        i += 1
    if receivers[j][1] < 0.01:
        j += 1

## 9. Summary of each person

In [ ]:
summary = pd.DataFrame({
    'Person': people,
    'Paid': [paid[p] for p in people],
    'Fair Share': [share[p] for p in people],
    'Balance': [balance[p] for p in people]
})

summary.round(2)

## 10. Total spending by category

In [ ]:
paid_df = df[df['status'] == 'paid']
category_total = paid_df.groupby('category')['amount'].sum().sort_values(ascending=False)
category_total

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(category_total.index, category_total.values)
plt.title('Spending by Category')
plt.xlabel('Category')
plt.ylabel('Amount (Rs.)')
plt.tight_layout()
plt.show()

## 11. Amount paid by each person

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(summary['Person'], summary['Paid'])
plt.title('Amount Paid by Each Person')
plt.xlabel('Person')
plt.ylabel('Amount (Rs.)')
plt.tight_layout()
plt.show()

## 12. Pending payment

In [ ]:
pending_df = df[df['status'] == 'unpaid']

if pending_df.empty:
    print('No pending payments')
else:
    print(pending_df[['id', 'description', 'amount', 'participants']].to_string(index=False))

## 13. Simple observations

- Hotel has the highest spending in this small sample.
- Alice paid the highest total amount.
- Bob and Carol need to pay Alice to settle the trip expenses.
- The hotel refund reduces the amount finally owed.
- The snacks expense is unpaid, so it is kept as a pending payment.

### Formula
**Balance = Amount Paid − Fair Share**

## 14. Conclusion

This project uses basic data science methods to solve a real-world expense sharing problem. Pandas is used for data handling, NumPy for simple numerical work, and Matplotlib for visualisation. The program can be improved later by adding a user interface, a database or online payments.